## Data Reading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('csv').option('inferSchema',True).option('header',True).load('/Volumes/workspace/default/volumes/BigMart Sales.csv')

In [0]:
df.display()

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight DOUBLE,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [0]:
df = spark.read.format('csv') \
    .schema(my_ddl_schema) \
    .option('header',True) \
    .load('/Volumes/workspace/default/volumes/BigMart Sales.csv')

In [0]:
df.display()

In [0]:
my_struct_schema = StructType([StructField('Item_Identifier', StringType(),True),
                 StructField('Item_Weight', StringType(),True),
                 StructField('Item_Fat_Content',StringType(),True),
                 StructField('Item_Visibility',StringType(),True),
                 StructField('Item_MRP',StringType(),True),
                 StructField('Outlet_Identifier',StringType(),True),
                 StructField('Outlet_Establishment_Year',StringType(),True), 
                 StructField('Outlet_Size',StringType(),True), 
                 StructField('Outlet_Location_Type',StringType(),True), 
                 StructField('Outlet_Type',StringType(),True), 
                 StructField('Item_Outlet_Sales',StringType(),True)])

In [0]:
df = spark.read.format('csv') \
    .schema(my_ddl_schema) \
    .option('header',True) \
    .load('/Volumes/workspace/default/volumes/BigMart Sales.csv')

In [0]:
df.printSchema()

##SELECT FUNCTION = sel()

In [0]:
df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).display()

##ALIAS FUNCTION = alias()

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

##FILTER = filter()

Scenario 1

In [0]:
df.filter(col('Item_Fat_Content') == 'Regular').display()

Scenario - 2

In [0]:
df.filter( (col('Item_Type') == 'Soft Drinks') & (col('Item_Weight') < 10) ).display()

Scenario 3

In [0]:
df.filter((col('Outlet_Location_Type').isin('Tier 1','Tier 2')) & (col('Outlet_Size').isNull())).display()

#Renaming the Column = withColumnRenamed()

In [0]:
df.withColumnRenamed("Item_Weight","Item_wt").display()

#withColumn() = Adding a column and Modifying the existing column

Scenario 1 - Creating a new column

In [0]:
df.withColumn('flag',lit("New")).display()

In [0]:
df.withColumn('Multiply',col('Item_Weight') * col('Item_MRP')).display()

Scenario - 2 => Modifying the values on the existing columns

In [0]:
df.withColumn('Item_Fat_content',regexp_replace(col('Item_Fat_content'),'Regular','Reg'))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_content'),'Low Fat','LF')).display()

#Type casting => cast()

In [0]:
df = df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

In [0]:
df = df.withColumn('Item_Weight',col('Item_Weight').cast(IntegerType()))

In [0]:
df.printSchema()

#Sorting/orderBy => asc(), desc()

Scenario - 1

In [0]:
df.sort(col('Item_Weight').desc()).display()

#### Sorting based on multiple columns - Scenario 2

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending = [0,1]).display()

#### limit()

In [0]:
df.limit(10).display()

#### drop()

#### Scenario 1 => dropping 1 column

In [0]:
df.drop(col('Item_Weight')).display()

#### scenario 2 => dropping multiple columns

In [0]:
df.drop(col('Item_Weight'),col('Item_Visibility')).display()

#### Drop Duplicates = dropDuplicates()

Scenario - 1 => Just dropping all the duplicates

In [0]:
df.dropDuplicates().display()

Scenario 2 => Dropping duplicates for particular column

In [0]:
df.drop_duplicates(subset=['Item_Identifier']).display()

Scenario - 2

In [0]:
df.dropDuplicates().display()

In [0]:
df.distinct().display()

##Distinct = Drop Duplicates

#Union & Union By Name

In [0]:
data1 = [(1,'Sid'),(2,'Kad')]

schema1 = 'id integer, Name String'

df1 = spark.createDataFrame(data1,schema1)

data2 = [(3,'Rahul'),(4,'Jas')]

schema2 = 'id Integer, Name String'

df2 = spark.createDataFrame(data2,schema2)

Union

In [0]:
df1.union(df2).display()

If I use unionByName() it will perform union based on the column names irrespective of order of columns

In [0]:
df1.unionByName(df2).display()

#String Functions

#InitCap()

In [0]:
df.select(initcap('Item_Type').alias('Item_Type')).display()

#upper()

In [0]:
df.select(upper('Item_Type').alias('Item_Type')).display()

#lower()

In [0]:
df.select(lower('Item_Type').alias('Item_Type')).display()

#upper()

In [0]:
df.select(upper('Item_Type').alias('Item_Type')).display()

###Date Functions


# 1) current_date()

In [0]:
df = df.withColumn('Current Date',current_date())

In [0]:
df.display()

# 2) Date_Add()

In [0]:
df = df.withColumn('Week_after',date_add('Current Date',7))

df.display()

In [0]:
df = df.withColumn('Week_Before',date_add('Current Date',-7))

In [0]:
df.display()

#3) datediff()

In [0]:
df = df.withColumn('Number of Days',datediff('Week_after','Week_Before'))

df.display()

#4) date_format()

In [0]:
df = df.withColumn('Week_Before',date_format('Week_Before','dd-MM-yyyy'))

#Handling Null Values

In [0]:
df.display()

#1) Dropping Nulls => dropna()

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset = ['Outlet_Size']).display()

#2) Filling Null values => fillna()
fillna() works well only for string datatypes

In [0]:
df.fillna("Not Available").display()

In [0]:
df = df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

In [0]:
df = df.fillna(0.0,subset = ["Item_Weight"]).display()

In [0]:
df.display()

#Split & Indexing
Splitting a column and using indexing to retrieve the value inside an array

In [0]:
df_split = df.withColumn('Types',split('Outlet_Type',' ')[1])

In [0]:
df_split.display()

#Explode
Exploding the array values of a column

Splitting the Outlet_type column in order to converting it to an array

In [0]:
df_split = df.withColumn('Outlet_Type',split('Outlet_Type'," "))

In [0]:
display(df_split)

In [0]:
df_exp = df_split.withColumn('Outlet_Type',explode('Outlet_Type'))

In [0]:
display(df_exp)

#array_contains() = 
Returns boolean values if a particular value is available in a array or not

In [0]:
df_split.withColumn('Type1_Flag',array_contains('Outlet_Type','Type1')).display()

#GroupBY

Scenario - 1 => Finding the total sales based on Item Type

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

Scenario - 2 => Performing average and renaming the column using alias

In [0]:
df.groupBy('Item_Type').agg((avg('Item_MRP')).alias('Average')).display()

Scenario 3 => Performing group by for more than 1 column

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_amount')).display()

Scenario - 4 => Performing different aggregations on same columns

In [0]:
df.groupBy('Item_Type','Outlet_Size') \
    .agg(sum('Item_MRP').alias('Total_MRP'),avg('Item_MRP').alias('Avg_MRP')).display()

#collect_list() => Groups the required column and gives the result in array format

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

In [0]:
df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book').alias('Books_read')).display()

#Pivot

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

#When-Otherwise

Scenario - 1

In [0]:
df = df.withColumn('Veg-Flag',when(col('Item_Type') == 'Meat','Non-Veg').otherwise('Veg'))

In [0]:
df.withColumn('Veg_expensive_flag',when(((col('Veg-Flag')=='Veg') & (col('Item_MRP') < 100)),'Veg_Inexpensive')\
    .when((col('Veg-Flag')=='Veg') & (col('Item_MRP') > 100),'Veg_Expensive')\
        .otherwise('Non-Veg')).display()

##JOINS

Inner Join

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.join(df2,df1['dept_id'] == df2['dept_id'],'inner').display()

Left Join => Performing Joins by selecting only specific columns

In [0]:
result = df1.join(df2,df1['dept_id'] == df2['dept_id'],'left')\
    .select(df1.emp_id,df1.emp_name,df1.dept_id,df2.department)

In [0]:
result.display()

Right Join

In [0]:
df1.join(df2,df1['dept_id'] == df2['dept_id'],'right').\
    select(df1.emp_id,df1.emp_name,df2.dept_id,df2.department).display()

Anti-Join => Returns only the unmatched row from the left table

In [0]:
df1.join(df2,df1['dept_id'] == df2['dept_id'],'Anti').\
    select(df1.emp_id, df1.emp_name, df1.dept_id).display()

In [0]:
df.display()

#Windowing Functions()

rank() => If I want to order based on ascending way

In [0]:
my_window = Window.orderBy('Item_Identifier')

df.withColumn('Rank',rank().over(my_window)).display()

If I want to order based on descending way

In [0]:
my_window = Window.orderBy(col('Item_Identifier').desc())

df.withColumn('Rank',rank().over(my_window)).display()

#dense_rank()

In [0]:
my_window = Window.orderBy('Item_Identifier')

df.withColumn('dense_rank',dense_rank().over(my_window))\
    .withColumn('rank',rank().over(my_window)).display()

#Cummulative Sum => sum()

In [0]:
my_window = Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow)
df.withColumn('Cumsum',sum('Item_MRP').over(my_window)).display()

In [0]:
my_window = Window.orderBy('Item_MRP').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)
df.withColumn('Total',sum('Item_MRP').over(my_window)).display()

#User Defined Functions [UDF]

In [0]:
def my_func(x):
    return x*x

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('New_col',my_udf('Item_MRP')).display()

In [0]:
df.write.format('csv')\
    .save('/Volumes/workspace/default/volumes/data.csv')

#Data Writing Modes => 
1) Append 2) Overwrite 3) Error 4) Ignore

#Append

In [0]:
df.write.format('csv')\
    .mode('append')\
        .save('/Volumes/workspace/default/volumes/data.csv/')

#Overwrite

In [0]:
df.write.format('csv')\
    .mode('overwrite')\
        .save('/Volumes/workspace/default/volumes/data.csv/')

In [0]:
df.write.format('parquet')\
    .mode(overwrite)\
        .save('/Volumes/workspace/default/volumes/data.csv/')